# self attention

###  Implementing self-attention with trainable weights

- The most notable difference is the `introduction of weight matrices that are updated during model training`. These trainable weight matrices are crucial so that the model (specifically, the attention module inside the model) can learn to produce `good` context vectors.

- We will implement the self-attention mechanism step by step by introducing the three trainable weight matrices Wq, Wk, and Wv. These three matrices are used to project the embedded input tokens, x(i), into query, key, and value vectors, respectively.

  ![9. Self Attention](../images/9.self-attention.png)


- In the first step of the self-attention mechanism with trainable weight matrices, we compute query (q), key (k), and value (v) vectors for input elements x. Similar to previous sections, we designate the second input, x(2), as the query input. The query vector q(2) is obtained via matrix multiplication between the input x(2) and the weight matrix Wq. Similarly, we obtain the key and value vectors via matrix multiplication involving the weight matrices Wk and Wv.

- `finally we obtain k, q, v for the input by the given weights of thek, q, v matrices`

# why key, query and value

- The terms “key,” “query,” and “value” in the context of attention mechanisms are
borrowed from the domain of information retrieval and databases, where similar concepts are used to store, search, and retrieve information.

- but most focussed thing is that, unlike normal search over db, here model needs to understand the given query and search it in the db as a key later find a value. we need attention here to establish relation over the query and key, later key to value.

- so what we do here is q to key relation, later (q->key) to value relation, this is how flow goes through it. (relations mostly established using the matrix multiplication).

In [40]:
import torch
inputs = torch.tensor(
 [[0.43, 0.15, 0.89], # Your (x^1)
 [0.55, 0.87, 0.66], # journey (x^2)
 [0.57, 0.85, 0.64], # starts (x^3)
 [0.22, 0.58, 0.33], # with (x^4)
 [0.77, 0.25, 0.10], # one (x^5)
 [0.05, 0.80, 0.55]] # step (x^6)
)
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [39]:
x2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2
input_shape, output_shape


(torch.Size([3]), 2)

-  In GPT-like models, the input and output dimensions are usually the same, for our simplicity we are using the different shapes.
-  we initialize the three weight matrices Wq, Wk, and Wv 

In [22]:
torch.manual_seed(123)

wq = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
wk = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
wv = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
# we would set requires_grad=True to update these matrices during model training

query = x2 @ wq
key = x2 @ wk
value = x2 @ wv

print(query)

tensor([0.4306, 1.4551])


# Weight parameters vs. attention weights 

- weights are for the model training or the network connections, while attention weights specifies that relation of context in entire text spaces.

- `attention weights determine the extent to which a context vector depends on the different parts of the input`.

- `the term “weight” is short for “weight parameters,” the values of a neural network that are optimized during training`.

- we calculate the keys and values by multiplying the tokens embedding of "the quick brown fox jumps over" with the K and V matrices.

In [51]:
keys = inputs @ wk
values = inputs @ wv
# queries = inputs @ wq
# query for single
keys, values,query , inputs.shape, wk.shape

(tensor([[0.3669, 0.7646],
         [0.4433, 1.1419],
         [0.4361, 1.1156],
         [0.2408, 0.6706],
         [0.1827, 0.3292],
         [0.3275, 0.9642]]),
 tensor([[0.1855, 0.8812],
         [0.3951, 1.0037],
         [0.3879, 0.9831],
         [0.2393, 0.5493],
         [0.1492, 0.3346],
         [0.3221, 0.7863]]),
 tensor([0.4306, 1.4551]),
 torch.Size([6, 3]),
 torch.Size([3, 2]))

In [52]:

keys.shape, values.shape
query

tensor([0.4306, 1.4551])

In [53]:
# now we need to find the attention scores as before , but here found by key and query, not only with the input as previous.
attention_score = query @ keys.T
attention_score

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])

- converting the attention scores to attention weights, by softmax and also with the scaling the attention scores.
  
- now we scale the attention scores by dividing them by the square root of the embedding dimension of the keys.

# why do we need to scale down even though using the softmax.

- improve training performance by avoiding small gradients. `Going back to the example of the w values, where w_1 has the largest value, because of the "winner-takes-most" effect, w_1 dominated the share of attention weights with a_1 = 99.32%. If w_1 is the wrong choice, the gradient (i.e. signal to "fix the mistake") will be nearly zero, and the model will not learn`.

- Instead, we now scale the values by dividing by the square root of d_in, and we assume the value of d_in was 25, and the square root is therefore 5:

w_1: 10/5 = 2
w_2: 5/5 = 1
w_3: 1/5 = 0.2

- Applying softmax, the share of w_1 is now e^2 / (e^2 + e^1 + e^0.2) = 0.652 (or 65.2%). As a result, most of the attention are still focused on w_1, but now more attention can also be paid to w_2 and w_3 to account for the possibility that w_1 could be wrong.

- For sake of completion, if we had simply divide by d_in instead of the square root, we get the following:

z_1: 10/25 = 0.5
z_2: 5/25 = 0.2
z_3: 1/25 = 0.04

- Applying softmax, the share of w_1 decreased even further to 0.0422 (42.2%). The share of w_2 increased from 24.0% to 31.2%, which may split the attention between w_1 and w_2 and and reduced attention on w_1 by a too large degree such that not enough attention is placed on w_1 (When we should have given that it has the largest value).

In [54]:
d_k = keys.shape[-1] # last index no of embeddings per each key
attention_wts = torch.softmax(attention_score/d_k**0.5, dim=-1)
attention_wts

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])

  ![9. Score scaling](../images/10.scaling.png)

  - now the final step is the context vectors by doing summing prod of `values and the attention_scores`.

In [55]:
context_vec = attention_wts @ values
context_vec

tensor([0.3061, 0.8210])

# Implementing a compact self-attention Python class

In [94]:
import torch.nn as nn
class selfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.wq = nn.Parameter(torch.rand(d_in, d_out))
        self.wk = nn.Parameter(torch.rand(d_in, d_out))
        self.wv = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
         # find all keys and values
        keys = inputs @ self.wk
        values = inputs @ self.wv
        queries = inputs @ self.wq

        # attention_scores
        attention_scores = queries @ keys.T

        # attention_weights by scaling and normalisation
        d_k = keys.shape[-1]
        attention_wts = torch.softmax(attention_scores/(d_k**0.5), dim=-1)

        # context vectors
        context_vec = attention_wts @ values

        return context_vec
        

In [95]:
torch.manual_seed(123)
sa_v1 = selfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


- Self-attention involves the trainable weight matrices Wq, Wk, and Wv. These matrices transform input data into queries, keys, and values, respectively, which are crucial components of the attention mechanism.
- As the model is exposed to more data during training, it adjusts these trainable weights.

  ![11. QKV relation](../images/11.QKVrelation.webp)

  ![12. QKV relation](../images/12.QKVrelation2.png)

- We can improve the SelfAttention_v1 implementation further by utilizing PyTorch’s nn.Linear layers
- instead of manually implementing `nn.Parameter(torch.rand(...))` is that `nn.Linear has an optimized weight initialization scheme`, contributing to `more stable and effective model training`.

In [101]:

class Self_Attention_v2(nn.Module):
    def __init__(self, d_in, d_out,qkv_bias=False):
        super().__init__()
        # initialise the weights but with nn.linear
        self.wq = nn.Linear(d_in,d_out, bias=qkv_bias)
        self.wk = nn.Linear(d_in,d_out, bias=qkv_bias)
        self.wv = nn.Linear(d_in,d_out, bias=qkv_bias)

    def forward(self, inputs):
        # get all keys, values and queries from input based on the weights
        queries = self.wq(inputs)
        keys = self.wk(inputs)
        values = self.wv(inputs)

        # attetion weights by keys and queries
        attention_scores = queries @ keys.T

        # attention scores after normalisation
        attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5, dim=-1)

        # final context vectors
        context_vectors = attention_weights @ values

        return context_vectors
        
        
        

In [102]:
torch.manual_seed(789)
sa_v2 = Self_Attention_v2(d_in, d_out)
print(sa_v1(inputs))
sa_v2(inputs)


tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)

- In above self_att1 and 2 has difference is initial weights assigning. using linear it is more efficient